In [12]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [13]:
import pandas as pd
import numpy as np

In [14]:
import importlib
import matrix_factorization

importlib.reload(matrix_factorization)

from matrix_factorization import train_matrix_factorization, recommend_mf

In [21]:
(   final_predictions,
    train_rmse,
    test_rmse,
    Y_df,
    R
) = train_matrix_factorization()

print(f"Train RMSE: {train_rmse:.4f}")
print(f"Test RMSE : {test_rmse:.4f}")

Train RMSE: 0.7699
Test RMSE : 0.8352


In [22]:
movie_means = np.nanmean(Y_df.values, axis=0)
baseline_predictions = np.tile(movie_means, (Y_df.shape[0], 1))

R_mask = Y_df.notna().values

np.random.seed(42)
test_mask = (R_mask == 1) & (np.random.rand(*R_mask.shape) < 0.2)

baseline_rmse = np.sqrt(
    np.mean(
        (
            baseline_predictions[test_mask]
            - Y_df.fillna(0).values[test_mask]
        ) ** 2
    )
)

In [24]:
results = pd.DataFrame({
    "Model": [
        "Movie Average Baseline",
        "Matrix Factorization"
    ],
    "RMSE": [baseline_rmse,test_rmse]})
results

,Model,RMSE
0,Movie Average Baseline,0.915695
1,Matrix Factorization,0.835206


In [18]:
def recommend_user(user_id, top_n=10):
    user_idx = user_id - 1

    return (
        pd.Series(
            final_predictions[user_idx],
            index=Y_df.columns
        )
        .drop(
            Y_df.columns[R[user_idx] == 1],
            errors="ignore"
        )
        .sort_values(ascending=False)
        .head(top_n)
    )

pd.DataFrame({
    "Movie": recommend_user(10).index,
    "Predicted Rating": recommend_user(10).values
})

,Movie,Predicted Rating
0,L.A. Confidential (1997),3.709757
1,"Departed, The (2006)",3.667929
2,Trainspotting (1996),3.562465
3,Wallace & Gromit: The Wrong Trousers (1993),3.508987
4,Raiders of the Lost Ark (Indiana Jones and the...,3.501824
5,Monty Python and the Holy Grail (1975),3.490244
6,Ferris Bueller's Day Off (1986),3.471257
7,Spirited Away (Sen to Chihiro no kamikakushi) ...,3.471100
8,"Shawshank Redemption, The (1994)",3.464937
9,Apocalypse Now (1979),3.462557


### Matrix Factorization Evaluation
**Latent Factor Collaborative Filtering**


A matrix factorization model was trained using gradient descent and regularization.

Evaluation was performed using RMSE on a held-out test set.

| Model | RMSE |
|--------|--------|
| Movie Average Baseline | 0.915695 |
| Matrix Factorization | 0.835206 |

The matrix factorization model achieved a lower RMSE than the baseline model, indicating better rating prediction performance.

Additionally, the model was able to generate personalized movie recommendations for users by predicting ratings for unseen movies.